# 1. A rectangular room, checked against the exact solution

A room resonates at its **modes**: standing waves at particular frequencies. In a rectangular
room with rigid walls they are known exactly,

$$f = \frac{c}{2}\sqrt{\left(\frac{n_x}{L}\right)^2 + \left(\frac{n_y}{W}\right)^2 + \left(\frac{n_z}{H}\right)^2},$$

so this room is the test of the finite element method (FEM) that we then use for rooms of any shape.

Modes matter below the **Schroeder frequency**: there they are few and far apart, and each one
can be heard. Above it they overlap so densely that the room is described by statistics
(the reverberation time).

In [ ]:
import itertools
import numpy as np
import plotly.graph_objects as go
from IPython.display import Audio

import acousticfem as af
from acousticfem import synth

## Modes up to 150 Hz

`element_size` chooses the mesh for the highest frequency: quadratic elements (`order=2`),
about four per wavelength, keep the error below 1%.

In [ ]:
L, W, H = 8.0, 6.0, 3.5
c = af.physics.speed_of_sound(20.0)
f_max = 150.0

room = af.box(L, W, H, af.element_size(f_max, c))
modes = af.compute(room, f_max, c)
print(f"{len(modes)} modes, {modes.info['dofs']} unknowns, "
      f"{modes.info['seconds_assembly'] + modes.info['seconds_eigensolver']:.1f} s")

In [ ]:
exact = np.sort([c / 2 * np.sqrt((a / L)**2 + (b / W)**2 + (d / H)**2)
                 for a, b, d in itertools.product(range(15), repeat=3)])
exact = exact[(exact > 0) & (exact <= f_max)]
n = min(len(modes), len(exact))
error = 100 * (modes.frequencies[:n] - exact[:n]) / exact[:n]
print(f"exact modes: {len(exact)}, found: {len(modes)}, largest error {np.abs(error).max():.2f} %")

go.Figure(go.Scatter(x=exact[:n], y=error, mode="markers")).update_layout(
    xaxis_title="Exact frequency [Hz]", yaxis_title="FEM error [%]", height=350)

The FEM slightly overestimates the frequencies, more at high frequency where the elements
are coarser compared with the wavelength. A mode just below `f_max` can therefore end up above
it and be left out.

## Materials and decay

Each surface gets a material. Every mode decays at its own rate, which depends on how much
of its energy touches the absorbing surfaces: here the floor absorbs, so modes with strong
pressure on the floor decay faster.

In [ ]:
print(list(af.list_materials())[:10], "...")
absorption = {"floor": "carpet_thick", "ceiling": "ceiling_plasterboard",
              "x0": "plaster_gypsum", "x1": "plaster_gypsum", "y0": "glass_window_heavy", "y1": "plaster_gypsum"}
modes.set_absorption(absorption, temperature=20, humidity=50)
stats = af.room_statistics(room.volume, room.areas(), absorption)
print(f"Schroeder frequency: {stats['schroeder_frequency_hz']:.0f} Hz")
print(f"T60 Sabine at 125 Hz: {stats['octave_bands'][0]['t60_sabine_s']:.2f} s")

In [ ]:
source, receiver = [1.2, 1.5, 1.2], [6.5, 4.4, 1.6]
af.plots.spectrum(modes, source, receiver, stats["schroeder_frequency_hz"])

In [ ]:
af.plots.gallery(modes, axis="z", position=1.2, max_modes=8)

## Listening to the modes

The modal part of the impulse response, from source to receiver. It contains only the modes,
so only the low frequencies: the boom of the room.

In [ ]:
phi = modes.evaluate([source, receiver])
ir = synth.impulse_response(2.0, 48000, modes.frequencies, modes.decay, phi[0], phi[1], c)
Audio(ir, rate=48000)